# <b><span style='color:#F1A424'>|</span> CMI: <span style='color:#F1A424'>Sequence Data Augmentation</span>
***

In this notebook I propose some data augmentations for this competition. Despite that our data is not sound sequences, some may still be applied if they are correctly adapted to our domain. Most of these functions are modified to work with our competition data which consists of short sequences (~100 steps) as arrays of length `L` and `C` channels.

Here I propose the following augmentations, many more can be crafted, if you want to share your augmentations please leave a comment below:
- GaussianNoise
- PinkNoiseSNR
- TimeStretch
- TimeShift
- ButterFilter
- MixUp

This notebook was inspired by the great [notebook][1] of Hidehisa Arai @hidehisaarai1213 !

I hope you learn something new from this work! 🙌🏼

### <b><span style='color:#F1A424'>Table of Contents</span></b> <a class='anchor' id='top'></a>
<div style=" background-color:#3b3745; padding: 13px 13px; border-radius: 8px; color: white">
    <li> <a href="#introduction">Introduction</a></li>
    <li> <a href="#install_libraries">Install libraries</a></li>
    <li><a href="#import_libraries">Import Libraries</a></li>
    <li><a href="#configuration">Configuration</a></li>
    <li><a href="#utils">Utils</a></li>
    <li><a href="#load_data">Load Data</a></li>
    <li><a href="#preprocessing">Data Pre-processing</a></li>
    <li><a href="#data_augmentation">Data Augmentation</a></li>
    <li><a href="#gaussian_noise">Gaussian Noise</a></li>
    <li><a href="#pink_noise">Pink Noise</a></li>
    <li><a href="#time_stretch">Time Stretch</a></li>
    <li><a href="#time_shift">Time Shift</a></li>
    <li><a href="#butter_filter">Butter Filter</a></li>
    <li><a href="#dataset">Dataset</a></li>
    <li><a href="#mixup">MixUp</a></li>
    
    
</div>

[1]: https://www.kaggle.com/code/hidehisaarai1213/rfcx-audio-data-augmentation-japanese-english

# <b><span style='color:#F1A424'>|</span> Install libraries</b><a class='anchor' id='install_libraries'></a> [↑](#top) 

***

In [1]:
!pip install colorednoise

# <b><span style='color:#F1A424'>|</span> Import libraries</b><a class='anchor' id='import_libraries'></a> [↑](#top) 

***

In [2]:
import colorednoise as cn
import gc
import librosa
import math
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import random
import time
import torch
import torch.nn as nn
import torch.nn.functional as F


from copy import deepcopy
from glob import glob
from plotly.subplots import make_subplots
from scipy.signal import butter, lfilter
from torch.utils.data import DataLoader, Dataset, Subset
from tqdm import tqdm

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
pio.renderers.default = 'iframe'

# <b><span style='color:#F1A424'>|</span> Configuration</b><a class='anchor' id='configuration'></a> [↑](#top) 

***

In [3]:
class config:
    MIXUP_PROB = 0.5

class paths:
    OUTPUT_DIR = "/kaggle/working/output"
    TEST_CSV = "/kaggle/input/cmi-detect-behavior-with-sensor-data/test.csv"
    TEST_DEMOGRAPHICS = "/kaggle/input/cmi-detect-behavior-with-sensor-data/test_demographics.csv"
    TRAIN_CSV = "/kaggle/input/cmi-detect-behavior-with-sensor-data/train.csv"
    TRAIN_DEMOGRAPHICS = "/kaggle/input/cmi-detect-behavior-with-sensor-data/train_demographics.csv"

# <b><span style='color:#F1A424'>|</span> Utils</b><a class='anchor' id='utils'></a> [↑](#top) 

***

In [4]:
def plot_dual_timeseries_px(x, x_aug, titles=['Original', 'Augmented']):
    """
    Alternative approach using plotly express with melted dataframes
    """
    L, N = x.shape
    cols = ["acc_x", "acc_y", "acc_z", "rot_w", "rot_x", "rot_y", "rot_z"]
    # Create dataframes for both arrays
    df1 = pd.DataFrame(x, columns=cols)
    df1['Time'] = np.arange(L)
    df1['Dataset'] = titles[0]
    df1_melted = df1.melt(id_vars=['Time', 'Dataset'], var_name='Channel', value_name='Value')

    L, N = x_aug.shape
    df2 = pd.DataFrame(x_aug, columns=cols)
    df2['Time'] = np.arange(L)
    df2['Dataset'] = titles[1]
    df2_melted = df2.melt(id_vars=['Time', 'Dataset'], var_name='Channel', value_name='Value')
    
    # Combine dataframes
    df_combined = pd.concat([df1_melted, df2_melted], ignore_index=True)
    
    # Create the plot
    fig = px.line(
        df_combined, 
        x='Time', 
        y='Value', 
        color='Channel',
        facet_col='Dataset',
        title='Time Series Comparison'
    )
    
    # Update layout
    fig.update_layout(
        height=600, width=1000,
        template='plotly_dark',
    )
    
    return fig
    

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed) 
    

def sep():
    print("—"*100)


label_to_num = {
    'Above ear - pull hair': 0,  # < ------- TARGETS START
    'Cheek - pinch skin': 1,
    'Eyebrow - pull hair': 2,
    'Eyelash - pull hair': 3,
    'Forehead - pull hairline': 4,
    'Forehead - scratch': 5,
    'Neck - pinch skin': 6,
    'Neck - scratch': 7,  # < ------- TARGETS END
    'Drink from bottle/cup': 8,  # < ------- NON-TARGETS START
    'Feel around in tray and pull out an object': 9,
    'Glasses on/off': 10,
    'Pinch knee/leg skin': 11,
    'Pull air toward your face': 12,
    'Scratch knee/leg skin': 13,
    'Text on phone': 14,
    'Wave hello': 15,
    'Write name in air': 16,
    'Write name on leg': 17  # < ------- NON-TARGETS END
}
type_to_num = {"Target": 1, "Non-Target": 0}
num_to_label = {v: k for k, v in label_to_num.items()}
num_to_type = {v: k for k, v in type_to_num.items()}

# <b><span style='color:#F1A424'>|</span> Load Data</b><a class='anchor' id='load_data'></a> [↑](#top) 

***

In [5]:
df_train = pd.read_csv(paths.TRAIN_CSV)
df_test = pd.read_csv(paths.TEST_CSV)
df_train_demographics = pd.read_csv(paths.TRAIN_DEMOGRAPHICS)
df_test_demographics = pd.read_csv(paths.TEST_DEMOGRAPHICS)

print(f"Train dataframe shape: {df_train.shape}"), sep()
print(f"Tesat dataframe shape: {df_test.shape}"), sep()
print(f"Train demographics dataframe shape: {df_train_demographics.shape}"), sep()
print(f"Test demographics dataframe shape: {df_test_demographics.shape}")

df_train["target"] = df_train["gesture"].map(label_to_num)
df_train["sequence_type"] = df_train["sequence_type"].map(type_to_num)

display(df_train.head())
display(df_train_demographics.head())

Train dataframe shape: (574945, 341)
————————————————————————————————————————————————————————————————————————————————————————————————————
Tesat dataframe shape: (107, 336)
————————————————————————————————————————————————————————————————————————————————————————————————————
Train demographics dataframe shape: (81, 8)
————————————————————————————————————————————————————————————————————————————————————————————————————
Test demographics dataframe shape: (2, 8)


,row_id,sequence_type,sequence_id,sequence_counter,subject,orientation,behavior,phase,gesture,acc_x,...,tof_5_v55,tof_5_v56,tof_5_v57,tof_5_v58,tof_5_v59,tof_5_v60,tof_5_v61,tof_5_v62,tof_5_v63,target
0,SEQ_000007_000000,1,SEQ_000007,0,SUBJ_059520,Seated Lean Non Dom - FACE DOWN,Relaxes and moves hand to target location,Transition,Cheek - pinch skin,6.683594,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1
1,SEQ_000007_000001,1,SEQ_000007,1,SUBJ_059520,Seated Lean Non Dom - FACE DOWN,Relaxes and moves hand to target location,Transition,Cheek - pinch skin,6.949219,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1
2,SEQ_000007_000002,1,SEQ_000007,2,SUBJ_059520,Seated Lean Non Dom - FACE DOWN,Relaxes and moves hand to target location,Transition,Cheek - pinch skin,5.722656,...,-1.0,112.0,119.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1
3,SEQ_000007_000003,1,SEQ_000007,3,SUBJ_059520,Seated Lean Non Dom - FACE DOWN,Relaxes and moves hand to target location,Transition,Cheek - pinch skin,6.601562,...,-1.0,101.0,111.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1
4,SEQ_000007_000004,1,SEQ_000007,4,SUBJ_059520,Seated Lean Non Dom - FACE DOWN,Relaxes and moves hand to target location,Transition,Cheek - pinch skin,5.566406,...,-1.0,101.0,109.0,125.0,-1.0,-1.0,-1.0,-1.0,-1.0,1


,subject,adult_child,age,sex,handedness,height_cm,shoulder_to_wrist_cm,elbow_to_wrist_cm
0,SUBJ_000206,1,41,1,1,172.0,50,25.0
1,SUBJ_001430,0,11,0,1,167.0,51,27.0
2,SUBJ_002923,1,28,1,0,164.0,54,26.0
3,SUBJ_003328,1,33,1,1,171.0,52,25.0
4,SUBJ_004117,0,15,0,1,184.0,54,28.0


# <b><span style='color:#F1A424'>|</span> Data pre-processing</b><a class='anchor' id='preprocessing'></a> [↑](#top) 

***

In [6]:
def standard_scale(arr: np.ndarray) -> np.ndarray:
    means = np.nanmean(arr, axis=0)
    stds = np.nanstd(arr, axis=0)
    stds = np.where(stds == 0, 1, stds)  # Prevent division by zero for constant columns
    scaled = (arr - means) / stds
    return scaled


def pad_or_truncate(
    arr: np.ndarray,
    max_length: int = 200,
    pad_value: int = 0,
    mode: str = "random"  # "regular" or "random"
) -> np.ndarray:
    L, D = arr.shape

    if L > max_length:
        return arr[:max_length, :]

    elif L < max_length:
        if mode == "regular":
            padding = np.full((max_length - L, D), pad_value)
            return np.vstack((arr, padding))
        
        elif mode == "random":
            total_padding = max_length - L
            pad_start = np.random.randint(0, total_padding + 1)
            pad_end = total_padding - pad_start

            start_padding = np.full((pad_start, D), pad_value)
            end_padding = np.full((pad_end, D), pad_value)

            return np.vstack((start_padding, arr, end_padding))
        
        else:
            raise ValueError(f"Unknown mode: {mode}. Use 'regular' or 'random'.")

    else:
        return arr


imu_cols = ["acc_x", "acc_y", "acc_z", "rot_w", "rot_x", "rot_y", "rot_z"]
all_X, all_y, all_y_hard = [], [], []
lengths = []
sequence_ids = df_train.sequence_id.unique()[0:100]

for sequence_id in tqdm(sequence_ids):
    ds = df_train[df_train["sequence_id"] == sequence_id]
    X = ds[imu_cols].values
    lengths.append(X.shape[0])
    y = ds.target.values[0]
    y_hard = ds.sequence_type.values[0]
    acc = standard_scale(X[:, 0:3])
    acc = np.where(np.isnan(acc), 0.0, acc)
    rot = X[:, 3:]
    X = np.concatenate([acc, rot], axis=1)
    X = np.where(np.isnan(X), 0.0, X)  # fill NaNs
    all_X.append(X)
    all_y.append(y)
    all_y_hard.append(y_hard)

all_y = np.array(all_y)
all_y_hard = np.array(all_y_hard)
print(len(X))

100%|██████████| 100/100 [00:04<00:00, 21.07it/s]

88


# <b><span style='color:#F1A424'>|</span> Data Augmentation</b><a class='anchor' id='data_augmentation'></a> [↑](#top) 

***

The following classes were inspired by the [albumentations][1] library. These are not our data augmentations per se but helper classes that will help us build our data augmentation pipeline.

- `SignalTransform`: A base class for all signal transformations (augmentations). It defines when and how a transform should be applied, using probabilistic control.
- `Compose`: A pipeline class that applies multiple transforms in sequence.
- `OneOf`: Randomly chooses one transform from a list and applies only that one.

[1]: https://albumentations.ai/docs/

In [7]:
class SignalTransform:
    def __init__(self, always_apply: bool = False, p: float = 0.5):
        self.always_apply = always_apply
        self.p = p

    def __call__(self, y: np.ndarray):
        if self.always_apply:
            return self.apply(y)
        else:
            if np.random.rand() < self.p:
                return self.apply(y)
            else:
                return y

    def apply(self, y: np.ndarray):
        raise NotImplementedError


class Compose:
    def __init__(self, transforms: list):
        self.transforms = transforms

    def __call__(self, y: np.ndarray):
        for trns in self.transforms:
            y = trns(y)
        return y


class OneOf:
    def __init__(self, transforms: list):
        self.transforms = transforms

    def __call__(self, y: np.ndarray):
        n_trns = len(self.transforms)
        trns_idx = np.random.choice(n_trns)
        trns = self.transforms[trns_idx]
        return trns(y)

# <b><span style='color:#F1A424'>|</span> Gaussian Noise</b><a class='anchor' id='gaussian_noise'></a> [↑](#top) 

***

The `AddGaussianNoise` class is a data augmentation transform for adding Gaussian (normally distributed) noise to a signal, often used in training models to improve robustness and generalization.

This class inherits from `SignalTransform`, meaning it benefits from the probabilistic behavior defined there (i.e., apply the transformation with a certain probability p).

This class randomly selects a noise scale between 0.0 and `max_noise_amplitude`. It then generates standard normal noise (mean 0, std 1) with the same shape as the input signal. Finally, it scales the noise by `noise_amplitude`, adds it to the signal, and casts the result back to the original data type.

In [8]:
class GaussianNoise(SignalTransform):
    def __init__(
        self, always_apply: bool = False, 
        p: float = 0.5, max_noise_amplitude: float = 0.20, **kwargs
    ):
        super().__init__(always_apply, p)
        self.noise_amplitude = (0.0, max_noise_amplitude)

    def apply(self, x: np.ndarray, **params):
        noise_amplitude = np.random.uniform(*self.noise_amplitude)
        noise = np.random.randn(*x.shape)  # shape (L, N)
        augmented = (x + noise * noise_amplitude).astype(x.dtype)
        return augmented

i = np.random.choice(range(len(X)))
x = all_X[i]
transforms = GaussianNoise(always_apply=True)
x_aug = transforms(x)

plot_dual_timeseries_px(x, x_aug)

# <b><span style='color:#F1A424'>|</span> Pink Noise</b><a class='anchor' id='pink_noise'></a> [↑](#top) 

***

This class adds **pink noise** to a multichannel time-series signal while maintaining a controlled signal-to-noise ratio (SNR).

Its purpose is to simulate realistic background noise in a signal by adding **pink noise** (a common type of noise in natural systems where power decreases with frequency) at a random SNR between `min_snr` and `max_snr`. The result is a noisy version of the input signal with pink noise at a controlled intensity.

Brief Explanation

1. Randomly selects an SNR from the specified range.
2. Computes the amplitude of the signal.
3. Calculates the noise amplitude needed to achieve the target SNR.
4. Generates pink noise for each channel using `cn.powerlaw_psd_gaussian(1, len(y))`.
5. Normalizes the pink noise to match the target amplitude.
6. Adds the scaled pink noise to the original signal.

In [9]:
class PinkNoiseSNR(SignalTransform):
    def __init__(self, always_apply=False, p=0.5, min_snr=5.0, max_snr=20.0, **kwargs):
        super().__init__(always_apply, p)
        self.min_snr = min_snr
        self.max_snr = max_snr

    def apply(self, y: np.ndarray, **params):
        snr = np.random.uniform(self.min_snr, self.max_snr)
        a_signal = np.sqrt((y ** 2).max(axis=0))  # shape: (N,)
        a_noise = a_signal / (10 ** (snr / 20))   # shape: (N,)
        pink_noise = np.stack([cn.powerlaw_psd_gaussian(1, len(y)) for _ in range(y.shape[1])], axis=1)
        a_pink = np.sqrt((pink_noise ** 2).max(axis=0))  # shape: (N,)
        pink_noise_normalized = pink_noise * (a_noise / a_pink)
        augmented = (y + pink_noise_normalized).astype(y.dtype)
        return augmented


i = np.random.choice(range(len(X)))
x = all_X[i]
transforms = PinkNoiseSNR(always_apply=True)
x_aug = transforms(x)

plot_dual_timeseries_px(x, x_aug)

# <b><span style='color:#F1A424'>|</span> Time Stretch</b><a class='anchor' id='time_stretch'></a> [↑](#top) 

***

This class changes the speed (duration) of a 1D or 2D signal by stretching or compressing it in the time dimension without changing the sampling rate.

Its purpose is to simulate variations in signal duration by speeding up or slowing down the signal, which helps improve model robustness to different timing or tempo variations in signals like audio, sensor data, or time series.

Brief Explanation:

1. Randomly selects a stretch factor (`rate`) between `min_rate` and `max_rate`.
2. Calculates the new length of the signal after stretching/compressing.
3. Uses linear interpolation to resample the original signal over the new time indices.
4. Supports both 1D (single channel) and 2D (multi-channel) input arrays.
5. Returns the time-stretched signal with shape adjusted according to the rate.

In [10]:
class TimeStretch(SignalTransform):
    def __init__(self, max_rate=1.5, min_rate=0.5, always_apply=False, p=0.5):
        super().__init__(always_apply, p)
        self.max_rate = max_rate
        self.min_rate = min_rate
        self.always_apply = always_apply
        self.p = p

    def apply(self, x: np.ndarray):
        """
        Stretch a 1D or 2D array in time using linear interpolation.
        - x: np.ndarray of shape (L,) or (L, N)
        - rate: float, e.g., 1.2 for 20% longer, 0.8 for 20% shorter
        """
        rate = np.random.uniform(self.min_rate, self.max_rate)
        L = x.shape[0]
        L_new = int(L / rate)
        orig_idx = np.linspace(0, L - 1, num=L)
        new_idx = np.linspace(0, L - 1, num=L_new)

        if x.ndim == 1:
            return np.interp(new_idx, orig_idx, x)
        elif x.ndim == 2:
            return np.stack([
                np.interp(new_idx, orig_idx, x[:, i]) for i in range(x.shape[1])
            ], axis=1)
        else:
            raise ValueError("Only 1D or 2D arrays are supported.")


i = np.random.choice(range(len(X)))
x = all_X[i]
transforms = TimeStretch(always_apply=True)
x_aug = transforms(x)

plot_dual_timeseries_px(x, x_aug)

# <b><span style='color:#F1A424'>|</span> Time Shift</b><a class='anchor' id='time_shift'></a> [↑](#top) 

***


This class randomly shifts a multichannel time-series signal along the time axis, simulating temporal misalignment or delay.

Its purpose is to introduce random temporal shifts in the input signal to make models more robust to timing variations or misalignments, commonly useful in audio, sensor, or sequential data.

Brief Explanation

1. Confirms the input is a 2D array with shape `(L, N)` where L is time length.
2. Computes the maximum allowed shift in samples based on `max_shift_pct`.
3. Randomly chooses a shift value between `-max_shift` and `+max_shift`.
4. Rolls (circularly shifts) the array along the time axis by the chosen amount.
5. If `padding_mode` is `"zero"`, replaces the wrapped-around shifted section with zeros to avoid artificial wraparound artifacts.
6. Returns the shifted signal.

In [11]:
class TimeShift(SignalTransform):
    def __init__(self, always_apply=False, p=0.5, max_shift_pct=0.25, padding_mode="replace"):
        super().__init__(always_apply, p)
        
        assert 0 <= max_shift_pct <= 1.0, "`max_shift_pct` must be between 0 and 1"
        assert padding_mode in ["replace", "zero"], "`padding_mode` must be either 'replace' or 'zero'"
        
        self.max_shift_pct = max_shift_pct
        self.padding_mode = padding_mode

    def apply(self, x: np.ndarray, **params):
        assert x.ndim == 2, "`x` must be a 2D array with shape (L, N)"
        
        L = x.shape[0]
        max_shift = int(L * self.max_shift_pct)
        shift = np.random.randint(-max_shift, max_shift + 1)

        # Roll along time axis (axis=0)
        augmented = np.roll(x, shift, axis=0)

        if self.padding_mode == "zero":
            if shift > 0:
                augmented[:shift, :] = 0
            elif shift < 0:
                augmented[shift:, :] = 0

        return augmented


i = np.random.choice(range(len(X)))
x = all_X[i]
transforms = TimeShift(always_apply=True)
x_aug = transforms(x)

plot_dual_timeseries_px(x, x_aug)

# <b><span style='color:#F1A424'>|</span> Butter Filter</b><a class='anchor' id='butter_filter'></a> [↑](#top) 

***

This class applies a low-pass Butterworth filter to multichannel time-series data.

Its purpose is to remove high-frequency components from the signal by applying a smooth low-pass filter, which helps reduce noise or unwanted rapid fluctuations, often used in sensor data preprocessing or audio denoising.

Brief Explanation:

1. Checks input is 2D with shape `(L, N)` (time length and channels).
2. Calls `butter_lowpass_filter` to process the signal.
3. Calculates normalized cutoff frequency relative to Nyquist frequency.
4. Designs Butterworth low-pass filter coefficients.
5. Applies the filter independently on each channel using `lfilter`.
6. Returns the filtered signal.

In [12]:
class ButterFilter(SignalTransform):
    def __init__(self, always_apply=False, p=0.5, cutoff_freq=20, sampling_rate=200, order=4):
        super().__init__(always_apply, p)
        
        self.cutoff_freq = cutoff_freq
        self.sampling_rate = sampling_rate
        self.order = order

    def apply(self, x: np.ndarray, **params):
        assert x.ndim == 2, "`x` must be a 2D array with shape (L, N)"
        return self.butter_lowpass_filter(x)

    def butter_lowpass_filter(self, data: np.ndarray):
        nyquist = 0.5 * self.sampling_rate
        normal_cutoff = self.cutoff_freq / nyquist
        b, a = butter(self.order, normal_cutoff, btype='low', analog=False)
        filtered_data = lfilter(b, a, data, axis=0)  # filter each channel independently
        return filtered_data


i = np.random.choice(range(len(X)))
x = all_X[i]
transforms = ButterFilter(always_apply=True)
x_aug = transforms(x)

plot_dual_timeseries_px(x, x_aug)

# <b><span style='color:#F1A424'>|</span> Composed</b><a class='anchor' id='composed'></a> [↑](#top) 

***

Now let's compose different augmentations into a single pipeline with different probabilities. In this example, only on of the three transformations inside the `OneOf` class is selected, if its probability is higher than `p` it is applied. Then the remaining transformations are applied if their probability is higher than `p`. Transformations are applied sequentially.

In [13]:
i = np.random.choice(range(len(X)))
x = all_X[i]
transforms = Compose([
    OneOf([
        GaussianNoise(p=0.5, max_noise_amplitude=0.05),
        PinkNoiseSNR(p=0.5, min_snr=4.0, max_snr=20.0),
        ButterFilter(p=0.5)
    ]),
    TimeStretch(p=0.25),
    TimeShift(p=0.25)

])
x_aug = transforms(x)

plot_dual_timeseries_px(x, x_aug)

# <b><span style='color:#F1A424'>|</span> Dataset</b><a class='anchor' id='dataset'></a> [↑](#top) 

***

Inside this `Dataset` class we will select and structure our sequence data and controll which transformations will be applied. In addition, `MixUp` is implemented inside the `__getitem__` method, which mixes two sequences data linearly and combines their respective ground truths accordingly as a weighted average.

In [14]:
class CustomDataset(Dataset):
    def __init__(
        self, config, df: pd.DataFrame, X: list[np.ndarray], y: list[np.ndarray], y_hard: list[np.ndarray],
        transforms = None, mode: str = "train"
    ): 
        self.config = config
        self.df = df
        self.X = X
        self.y = y
        self.y_hard = y_hard
        self.indexes = self.df.sequence_id.unique()[0:100]
        self.alpha = 0.3
        self.mode = mode
        self.transforms = transforms
        self.num_classes = 18
        
    def __len__(self):
        """
        Length of dataset.
        """
        return len(self.indexes)
        
    def __getitem__(self, i):
        """
        Get one item.
        """
        sequence_id = self.indexes[i]
        p = np.random.rand()
        if p <= 0.0:
            if self.mode == "train":
                X, y, y_hard = self.get_data(i)
                X = self.transforms(X)
                X = pad_or_truncate(X)
            else:
                X, y, y_hard = self.get_data(i)
                X = pad_or_truncate(X)
        else:
            if self.mode == "train":
                lam = np.random.beta(self.alpha, self.alpha)
                j = np.random.randint(0, len(self.indexes))
                X1, y1, y1_hard = self.get_data(i)
                X2, y2, y2_hard = self.get_data(j)
                X1, X2 = pad_or_truncate(X1), pad_or_truncate(X2) 
                X = lam * X1 + (1 - lam) * X2
                y = lam * y1 + (1 - lam) * y2
                y_hard = lam * y1_hard + (1 - lam) * y2_hard
            else:
                X, y, y_hard = self.get_data(i)
                X = pad_or_truncate(X)
            
        output = {
            "X": torch.tensor(X, dtype=torch.float32),
            "y": torch.tensor(y, dtype=torch.float32),
            "y_hard": torch.tensor(y_hard, dtype=torch.float32),
            "sequence_id": sequence_id
        }
        return output

    def get_data(self, index):
        X = self.X[index]
        y = self.y[index]
        y = np.eye(self.num_classes, dtype=int)[y]
        y_hard = self.y_hard[index]
        return X, y, y_hard
        

# <b><span style='color:#F1A424'>|</span> MixUp</b><a class='anchor' id='mixup'></a> [↑](#top) 

***

In [15]:
train_dataset = CustomDataset(config, df_train, all_X, all_y, all_y_hard, transforms, mode="train")
train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=0,
    pin_memory=True, drop_last=True
)
idx = np.random.choice(range(0, len(train_dataset)))
cols = imu_cols
sample = train_dataset[idx]
X = sample["X"]
y = sample["y"]
y_hard = sample["y_hard"]
print(f"Mixup y_true: {y}"), sep()
print(f"Mixup y_true_hard: {y_hard}"), sep()
sequence_id = train_dataset[idx]["sequence_id"]
N = X.shape[0]
df = pd.DataFrame(X, columns=cols)
df['step'] = range(N)
df_melted = df.melt(id_vars='step', var_name='sequence', value_name='value')

fig = px.line(
    df_melted,
    x='step',
    y='value',
    color='sequence',
    title=f"Sequences for {sequence_id}",
    template='plotly_dark',
    width=800,
    height=600
)

fig.show()

Mixup y_true: tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.8442, 0.1558, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000])
————————————————————————————————————————————————————————————————————————————————————————————————————
Mixup y_true_hard: 1.0
————————————————————————————————————————————————————————————————————————————————————————————————————
